<h1>Chapter 10 - Creating Text Embedding Models</h1>
<i>Exploring methods for both training and fine-tuning embedding models.</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter10/Chapter%2010%20-%20Creating%20Text%20Embedding%20Models.ipynb)

---

This notebook is for Chapter 10 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [1]:
# %%capture
# !pip install -q accelerate>=0.27.2 peft>=0.9.0 bitsandbytes>=0.43.0 transformers>=4.38.2 trl>=0.7.11 sentencepiece>=0.1.99
# !pip install -q sentence-transformers>=3.0.0 mteb>=1.1.2 datasets>=2.18.0

# Creating an Embedding Model

## **Data**

In [2]:
from datasets import load_dataset

# Load MNLI dataset from GLUE
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))

# Quiero el benchmarch GLUE, concretamente el dataset MNLI y selcciona los primeros 50000 ejemplos del conjunto de entrenamiento
# MNLI contiene pares de frases. Cada ejemplo tiene fundamentalmente:
    # {
    #    "premise": "...",
    #    "hypothesis": "...",
    #    "label": ... -> Puede ser 0 (entailment: la hipótesis se deduce de la premisa), 1 (neutral) o 2 (contradiction) 
    #}

train_dataset = train_dataset.remove_columns("idx")
# MNLI también trae una columna idx, que simplemente identifica el ejemplo -> No lo necesitamos -> Lo eliminamos

```text
GLUE
 └── MNLI
      └── train
           └── primeras 50.000 muestras
```

In [3]:
train_dataset[2]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

## **Model**

In [4]:
from sentence_transformers import SentenceTransformer

# Use a base model
embedding_model = SentenceTransformer('bert-base-uncased')
# bert-base-uncased no es originalmente un modelo Sentence-BERT entrenado para generar buenos embeddings de frases. Es BERT base.
# Al pasarle ese modelo a SentenceTransformer ->+  Quiero utilizar BERT como base para construir mi modelo de embeddings.

No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.
C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


La librería detecta que es un BERT normal y automáticamente construye una arquitectura nueva:

```text
BERT
  ↓
embeddings de tokens
  ↓
mean pooling
  ↓
embedding de frase
```

## **Loss Function**

Cómo utilizamos las etiquetas de MNLI para entrenar nuestro BERT + mean pooling?

In [5]:
from sentence_transformers import losses
# Importamos las distintas funciones de pérdida que proporciona sentence-transformers.
# La loss es la que va a decir durante el entrenamiento: Con los embeddings que estás generando, te estás equivocando tanto
# Y a partir de ese error, backpropagation modifica los pesos del modelo

# Define the loss function. In soft-max loss, we will also need to explicitly set the number of labels.
train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3
)
# Elegimos concretamente SoftmaxLoss
# SoftmaxLoss utiliza esos embeddings (de la premisa y de la hipótesis) para intentar predecir: entailment / neutral / contradiction
# trabaja con embeddings de ambas (u, v) y tb con |u-v|
# model=embedding_model -> Indica el modelo que queremos entrenar
# embedding_model.get_sentence_embedding_dimension()-> Toma las dimensiones que tiene el embedding que produce el modelo (768)
# num_labels=3 porque en MNLI tebenis 3 posibles etiquetas:
    # 0 → entailment
    # 1 → neutral
    # 2 → contradiction


## Evaluation

Estamos preparando cómo vamos a evaluar si el entrenamiento realmente mejora nuestros embeddings

In [6]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
# EmbeddingSimilarityEvaluator -> valúa una idea muy sencilla: Si dos frases son consideradas similares por humanos, 
# ¿sus embeddings también resultan similares?

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
# Seguimos dentro de GLUE oeri ahora no usamos MNLI sino STS-B
# (Semantic Textual Similarity Benchmark) contiene pares de frases junto con una puntuación humana de similitud. -> Puntuación de 0 a 5

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine",
)
# Crea el evaluador:
# sentences1 -> le pasamos todas las primeras frases sentences1[0], sentences1[1] ...
# sentences2 -> le pasamos todas las primeras frases sentences2[0], sentences2[1] ...
# Cada fila es un par que debe compararse
# scores -> El evaluador espera puntuaciones normalizadas, así que Alammar las divide entre 5
# main_similarity="cosine" -> Queremos evaluar los embeddings mediante similitud coseno.

```text
GLUE
├── MNLI
│    └── train → entrenamiento
│
└── STS-B
     └── validation → evaluación
```

## **Training**

In [7]:
# Definimos cómo vamos a ejecutar el entrenamiento:

from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir=r"D:\AI\base_embedding_model", # Carpeta donde se guardarán los resultados/checkpoints del entrenamiento
    num_train_epochs=1, # número de rondas de entrenamiento (veces que recorre todo el conjunto de entrenamiento)
    per_device_train_batch_size=32, # dividimos los ejemplos de entrenamiento (50.000) en batches de 32-> 50.000/32 = 1.563 pasos de entrenamiento
    per_device_eval_batch_size=32, # Igual pero para ejemplos de evaluación
    warmup_steps=100, # número de pasos en los que la LR irá creciendo linealmente hasta llegar a la LR definida 
    fp16=True, # ntrenamiento con precisión de 16 bits en lugar de utilizar siempre FP32
    eval_steps=100, # frecuencia de evaluación de cada 100 pasos, cuando la estrategia/configuración de evaluación hace uso de evaluación por pasos.
    logging_steps=100, # cada cuanto muestra información del entrenamiento
)

- per_device_eval_batch_size=32 → CÓMO procesar los datos al evaluar
- eval_steps=100 → CUÁNDO evaluar

c omo tenemos aproximadamente 1.563 pasos de entrenamiento, potencialmente habría evaluaciones alrededor de los pasos 100, 200, 300, ... 1.500, siempre que la configuración del Trainer que viene a continuación active efectivamente la evaluación por pasos

In [9]:
# Se crea el entrenador y se lanza el entrenamiento

from sentence_transformers.trainer import SentenceTransformerTrainer
#Importamos el Trainer específico de sentence-transformers

# Train embedding model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
# Cada argumento corresponde a algo que ya hemos definido

trainer.train()
# empieza el entrenamiento

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step,Training Loss
100,1.075900
200,0.942600
300,0.880500
400,0.851200
500,0.833100
600,0.848000
700,0.826100
800,0.818100
900,0.808500
1000,0.799700


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=1563, training_loss=0.8320552511087077, metrics={'train_runtime': 216.7519, 'train_samples_per_second': 230.678, 'train_steps_per_second': 7.211, 'total_flos': 0.0, 'train_loss': 0.8320552511087077, 'epoch': 1.0})

```text
model = embedding_model
→ bert-base-uncased + mean pooling

args = args
→ epochs, batch size, fp16, warmup, eval_steps...

train_dataset = train_dataset
→ 50.000 muestras de MNLI

loss = train_loss
→ SoftmaxLoss con 3 clases

evaluator = evaluator
→ STS-B + cosine similarity
```

In [10]:
# Evaluate our trained model
# Después de entrenar el modelo, lo evaluamos
evaluator(embedding_model)

{'pearson_cosine': 0.414520860348514,
 'spearman_cosine': 0.5120161126791356,
 'pearson_manhattan': 0.47032921328708316,
 'spearman_manhattan': 0.5109941617968489,
 'pearson_euclidean': 0.4576396654535937,
 'spearman_euclidean': 0.5057715745674346,
 'pearson_dot': 0.3890725755337151,
 'spearman_dot': 0.41667584234220983,
 'pearson_max': 0.47032921328708316,
 'spearman_max': 0.5120161126791356}

El `evaluator` usa **STS-B**:

```text
sentence1
sentence2
score humano (0–1)
```

Para cada pareja:

```text
sentence1 → embedding_model → vector 1
sentence2 → embedding_model → vector 2
                         ↓
                 cosine similarity
                         ↓
          comparar con el score humano
```

Aquí ya **no intervienen `SoftmaxLoss` ni las clases de MNLI**: solo se usaron durante el entrenamiento.

La evaluación mide la **correlación** entre:

- similitud coseno del modelo;
- puntuación humana de STS-B.

```text
correlación más alta → mejor modelo
```

# MTEB

In [13]:
from mteb import MTEB

# Choose evaluation task
evaluation = MTEB(tasks=["Banking77Classification"])
# Evalúa nuestro embedding_model en Banking77Classification
# Banking77Classification es una tarea de clasificación de intenciones en consultas bancarias. Hay 77 categorías/intenciones diferentes.

# Calculate results
results = evaluation.run(embedding_model)
results

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Classification

- Banking77Classification, s2s

[MTEBResults(task_name=Banking77Classification, scores=...)]

In [14]:
results[0].scores

{'test': [{'accuracy': 0.5049675324675326,
   'f1': 0.5027759896270394,
   'f1_weighted': 0.5027759896270394,
   'scores_per_experiment': [{'accuracy': 0.5038961038961038,
     'f1': 0.5024098009578631,
     'f1_weighted': 0.502409800957863},
    {'accuracy': 0.49642857142857144,
     'f1': 0.4968101742165161,
     'f1_weighted': 0.49681017421651597},
    {'accuracy': 0.5162337662337663,
     'f1': 0.5133221640169879,
     'f1_weighted': 0.5133221640169879},
    {'accuracy': 0.5301948051948052,
     'f1': 0.528199594815798,
     'f1_weighted': 0.5281995948157979},
    {'accuracy': 0.4987012987012987,
     'f1': 0.4949690110118394,
     'f1_weighted': 0.49496901101183943},
    {'accuracy': 0.5087662337662338,
     'f1': 0.5081104280024503,
     'f1_weighted': 0.5081104280024503},
    {'accuracy': 0.5074675324675325,
     'f1': 0.5040505224506198,
     'f1_weighted': 0.5040505224506197},
    {'accuracy': 0.48798701298701297,
     'f1': 0.4869904907265057,
     'f1_weighted': 0.4869904907

⚠️ **VRAM Clean-up** - You will need to run the code below to partially empty the VRAM (GPU RAM). If that does not work, it is advised to restart the notebook instead. You can check the resources on the right-hand side (if you are using Google Colab) to check whether the used VRAM is indeed low. You can also run `!nivia-smi` to check current usage.

In [ ]:
# # Empty and delete trainer/model
# trainer.accelerator.clear()
# del trainer, embedding_model

# # Garbage collection and empty cache
# import gc
# import torch

# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
# Esto es una alternativa pero no tan completa como reinciar el kernel
import gc
import torch

gc.collect()
# fuerza al recolector de basura de Python a eliminar objetos que ya no tienen referencias.

torch.cuda.empty_cache()
# devuelve a CUDA memoria que PyTorch tenía reservada en caché pero que ya no está siendo utilizada.

Para liberar VRAM:

```python
del trainer
del train_loss
del embedding_model

import gc
import torch

gc.collect()
torch.cuda.empty_cache()
```

Aun así:

```text
Restart Kernel
      >
del + gc.collect() + torch.cuda.empty_cache()
```

`Restart Kernel` libera de forma más completa toda la memoria asociada al proceso.

# Loss Fuctions

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

## Cosine Similarity Loss

Aquí MNLI se adapta para **`CosineSimilarityLoss`**.

Antes:

```text
0 → entailment
1 → neutral
2 → contradiction
```

Ahora:

```python
mapping = {2: 0, 1: 0, 0: 1}
```

```text
MNLI original        Nueva etiqueta

entailment      0  →  1.0   similares
neutral         1  →  0.0   no similares
contradiction   2  →  0.0   no similares
```

`CosineSimilarityLoss` ya no predice **3 clases**.

Quiere aprender una **puntuación de similitud**:

```text
1.0 → frases similares
0.0 → frases no similares
```

In [2]:
from datasets import Dataset, load_dataset

# Load MNLI dataset from GLUE
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

# (neutral/contradiction)=0 and (entailment)=1
mapping = {2: 0, 1: 0, 0:1}
train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [float(mapping[label]) for label in train_dataset["label"]]
})
# float(mapping[label]) hace que las etiquetas sean: 1.0 y 0.0
# Dataset.from_dict()  crea un objeto Dataset de Hugging Face a partir de un diccionario de Python
    # 1) Crea un diccionario
    # 2) Lo convierte en un datasets.Dataset estructurtado como una tabla donde_
        # Claves del diccionario -> columnas
        # elementos de listas -> Filas

Está construyendo un nuevo dataset con esta forma:

| `sentence1` | `sentence2` | `label` |
|---|---|---:|
| premise A | hypothesis A | 1.0 |
| premise B | hypothesis B | 0.0 |
| premise C | hypothesis C | 0.0 |

In [3]:
# Creamos el evaluador, el mismo que antes

from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [4]:
# Seguimos los mismos pasos que antes pero cambiamos las loss function a losses.CosineSimilarityLoss

from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="cosineloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.
C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step,Training Loss
100,0.231100
200,0.172200
300,0.170800
400,0.159200
500,0.153100
600,0.158800
700,0.151900
800,0.157500
900,0.148500
1000,0.146600


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=1563, training_loss=0.15722033158373697, metrics={'train_runtime': 287.7242, 'train_samples_per_second': 173.778, 'train_steps_per_second': 5.432, 'total_flos': 0.0, 'train_loss': 0.15722033158373697, 'epoch': 1.0})

In [5]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.7245711646583454,
 'spearman_cosine': 0.7255607242715452,
 'pearson_manhattan': 0.7338862274864126,
 'spearman_manhattan': 0.7329071599398508,
 'pearson_euclidean': 0.7335297366890552,
 'spearman_euclidean': 0.7320676308914783,
 'pearson_dot': 0.6789736223508394,
 'spearman_dot': 0.6802193567223545,
 'pearson_max': 0.7338862274864126,
 'spearman_max': 0.7329071599398508}

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## Multiple Negatives Ranking Loss

```text
DATOS DE PARTIDA (MNLI, solo entailment)

Premise 1 ───────── Hypothesis 1
    A1                   P1
              ✓

Premise 2 ───────── Hypothesis 2
    A2                   P2
              ✓

Premise 3 ───────── Hypothesis 3
    A3                   P3
              ✓

        ↓ barajamos las hypotheses

TRIPLETS DE ENTRENAMIENTO

              P1 (Positive)
             ↗
        acercar
           A1
        alejar
             ↘
              P3 (Negative)

        (A1, P1, P3)


              P2 (Positive)
             ↗
        acercar
           A2
        alejar
             ↘
              P1 (Negative)

        (A2, P2, P1)
```
### ¿Qué hace MNR?

```text
                  CANDIDATOS

               P1      P2      P3
              ┌──────┬──────┬──────┐
Anchor A1     │ 0.82 │ 0.21 │ 0.35 │  ← queremos máximo P1
              ├──────┼──────┼──────┤
Anchor A2     │ 0.19 │ 0.91 │ 0.27 │  ← queremos máximo P2
              ├──────┼──────┼──────┤
Anchor A3     │ 0.31 │ 0.25 │ 0.87 │  ← queremos máximo P3
              └──────┴──────┴──────┘
                 ↑       ↑       ↑
              positivos en la diagonal
```

**MNR (`MultipleNegativesRankingLoss`)** busca:

```text
A1 → acercar a P1 y alejar de P2, P3
A2 → acercar a P2 y alejar de P1, P3
A3 → acercar a P3 y alejar de P1, P2
```

Los demás positivos del batch actúan como **negativos para cada anchor**.


- Este bloque prepara MNLI para entrenar con MultipleNegativesRankingLoss, construyendo explícitamente triplets: (anchor, positive, negative)

In [2]:
import random
from tqdm import tqdm
from datasets import Dataset, load_dataset

# # Load MNLI dataset from GLUE
mnli = load_dataset("glue", "mnli", split="train").select(range(50_000))
# arga los primeros 50.000 ejemplos, igual que antes

mnli = mnli.remove_columns("idx")
# elimina columna "idx"

mnli = mnli.filter(lambda x: True if x['label'] == 0 else False)
# Se queda solo con los entailment (label->0 (entailment) -> Para MNR necesitamos pares positivos confiables: premisa + hipótesis
# Prepare data and add a soft negative
train_dataset = {"anchor": [], "positive": [], "negative": []}
# Crea un diccionario vacío con las 3 columnas que queremos construir
soft_negatives = mnli["hypothesis"]
random.shuffle(soft_negatives)
# para escoger la negativa toma primero las hipótesis positivas y las desordena aleatoriamente
# soft negatives porque no están escogidos como negativos especialmente difíciles ni verificados. 
# Simplemente son hypotheses de otros ejemplos escogidas por el shuffle
for row, soft_negative in tqdm(zip(mnli, soft_negatives)):
    train_dataset["anchor"].append(row["premise"])
    train_dataset["positive"].append(row["hypothesis"])
    train_dataset["negative"].append(soft_negative)
# Va recorriendo simultáneamente
    # row: ejemplo original de MNLI
    # soft_negative: una hipotesis aleatorio de la lista barajada
train_dataset = Dataset.from_dict(train_dataset)
# convierte el diccionario en un Dataset de Hugging Face:
len(train_dataset)
# Cuántos triplets se han creado

16875it [00:01, 15193.98it/s]


16875

El dataset quedaría con este formato:

| `anchor` | `positive` | `negative` |
|---|---|---|
| P1 | H1 | H3 |
| P2 | H2 | H1 |
| P3 | H3 | H4 |

In [3]:
# Evaluación, igual que en el caso de antes

from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [5]:
# Ejecutamos el experimento con MultipleNegativesRankingLoss

from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir=r"D:\AI\mnrloss_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.
C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step,Training Loss
100,0.340100
200,0.110000
300,0.086800
400,0.061000
500,0.071600


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=528, training_loss=0.1301089387499925, metrics={'train_runtime': 107.2303, 'train_samples_per_second': 157.372, 'train_steps_per_second': 4.924, 'total_flos': 0.0, 'train_loss': 0.1301089387499925, 'epoch': 1.0})

In [6]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.8011133179702983,
 'spearman_cosine': 0.8044565686838671,
 'pearson_manhattan': 0.816267990329934,
 'spearman_manhattan': 0.8119773283157369,
 'pearson_euclidean': 0.8159119871808314,
 'spearman_euclidean': 0.8117283904217435,
 'pearson_dot': 0.7101277243790574,
 'spearman_dot': 0.6983207855683604,
 'pearson_max': 0.816267990329934,
 'spearman_max': 0.8119773283157369}

In [7]:
import os

print(os.getcwd())
print(os.path.abspath("cosineloss_embedding_model"))

D:\Repos Github\Hands-On-Large-Language-Models\chapter10
D:\Repos Github\Hands-On-Large-Language-Models\chapter10\cosineloss_embedding_model


# **Fine-tuning**

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## **Supervised**

- Antes usamos SentenceTransformer("bert-base-uncased"), que es un BERT preentrenado en lenguaje, pero no es todavía un buen modelo especializado en sentence embeddings.
- SentenceTransformers le añadía pooling y nosotros lo entrenábamos para conseguir buenos embeddings.
- Ahora vamos a usar SentenceTransformer("sentence-transformers/**all-MiniLM-L6-v2**") que es un modelo de embeddings competente
- A éste le hacemos fine-tuning con MNLI + MNR y obtendremos un modelo de embeddings adaptado/mejorado
- **Hacemos lo de antes pero cambiando el punto de partida**

In [1]:
from datasets import load_dataset
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Load MNLI dataset from GLUE
# 0 = entailment, 1 = neutral, 2 = contradiction
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50_000))
train_dataset = train_dataset.remove_columns("idx")

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [2]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="finetuned_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\AI\HuggingFace\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Step,Training Loss
100,0.156700
200,0.110900
300,0.118800
400,0.117200
500,0.107000
600,0.102300
700,0.115200
800,0.100300
900,0.111100
1000,0.102100


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=1563, training_loss=0.1098792880907016, metrics={'train_runtime': 134.7716, 'train_samples_per_second': 370.998, 'train_steps_per_second': 11.597, 'total_flos': 0.0, 'train_loss': 0.1098792880907016, 'epoch': 1.0})

In [3]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.8485630381708795,
 'spearman_cosine': 0.8484731923019715,
 'pearson_manhattan': 0.8511893308707704,
 'spearman_manhattan': 0.8475675404314198,
 'pearson_euclidean': 0.852253349592617,
 'spearman_euclidean': 0.8484731923019715,
 'pearson_dot': 0.848563038488666,
 'spearman_dot': 0.8484731923019715,
 'pearson_max': 0.852253349592617,
 'spearman_max': 0.8484731923019715}

In [ ]:
# Evaluate the pre-trained model
original_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
evaluator(original_model)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'pearson_cosine': 0.8696194608752055,
 'spearman_cosine': 0.8671637433378804,
 'pearson_manhattan': 0.8670399009851635,
 'spearman_manhattan': 0.8663946139224048,
 'pearson_euclidean': 0.867871599362501,
 'spearman_euclidean': 0.8671643653432983,
 'pearson_dot': 0.8696194616795601,
 'spearman_dot': 0.8671631197908374,
 'pearson_max': 0.8696194616795601,
 'spearman_max': 0.8671643653432983}

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## **Augmented SBERT**

**Step 1:** Fine-tune a cross-encoder

In [1]:
# Este bloque prepara el gold dataset y lo deja en dos formatos distintos porque luego lo van a usar para cosas diferentes.

import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import InputExample
from sentence_transformers.datasets import NoDuplicatesDataLoader

# Prepare a small set of 10000 documents for the cross-encoder 
dataset = load_dataset("glue", "mnli", split="train").select(range(10_000))
# Una simulación: tenemos datos anotados limitados: engo pocos datos anotados fiables. -> Estos 10.000 -> gold dataset
mapping = {2: 0, 1: 0, 0:1} 
# mapeamos MNLI para que tenga el formato del cross-encoder
    # entailment (0) -> 1 (par positivo)
    # neutral y contradiction (1,2) -> 0 (par negativo)


# Data Loader
gold_examples = [
    InputExample(texts=[row["premise"], row["hypothesis"]], label=mapping[row["label"]])
    for row in tqdm(dataset)
]
# Aquí cada fila pasa a ser un InputExample de SentenceTransformers:
    #texts = [premise, hypothesis]
    # label = 0 o 1

gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)
# crea el DataLoader que irá entregando esos ejemplos en batches de 32.
# La parte de NoDuplicatesDataLoader intenta evitar que dentro de un mismo batch aparezcan textos duplicados. 
# Esto puede ser útil en entrenamiento contrastivo/por pares porque evita combinaciones redundantes o ambiguas.
# DataLoader decide cómo se los vamos entregando al modelo —batching, orden, evitar duplicados en este caso, etc.

# Pandas DataFrame for easier data handling
gold = pd.DataFrame(
    {
    'sentence1': dataset['premise'],
    'sentence2': dataset['hypothesis'],
    'label': [mapping[label] for label in dataset['label']]
    }
)
# Aquí construye un DataFrame de pandas con los mismos datos
# ¿Por qué tener ambas cosas? -> Cumplen funciones disintas:
    # gold_examples + gold_dataloader -> formato cómodo para ENTRENAR el cross-encoder
    # gold DataFrame -> formato cómodo para manipular, combinar, inspeccionar y ampliar datos después

100%|██████████| 10000/10000 [00:00<00:00, 25493.62it/s]


```text
MNLI 10.000
   ↓
mapping 0/1
   ↓
GOLD DATASET
   ├── InputExample + DataLoader → entrenamiento
   └── pandas DataFrame          → manipulación de datos
```

In [2]:
# Este bloque realiza el paso 1 de Augmented SBERT: entrenar el cross-encoder "profesor" con los datos Gold.

from sentence_transformers.cross_encoder import CrossEncoder

# Train a cross-encoder on the gold dataset
cross_encoder = CrossEncoder('bert-base-uncased', num_labels=2)
cross_encoder.fit(
    train_dataloader=gold_dataloader, # necesitamos siempre un dataloader
    epochs=1,
    show_progress_bar=True,
    warmup_steps=100,
    use_amp=False
)

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/312 [00:00<?, ?it/s]

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


**Step 2:** Create new sentence pairs

Ya hemos entrenado el cross-encoder con los 10.000 ejemplos gold y ahora vamos a preparar ejemplos sin etiquetar para que el cross-encoder los etiquete.

In [3]:
# Prepare the silver dataset by predicting labels with the cross-encoder
silver = load_dataset("glue", "mnli", split="train").select(range(10_000, 50_000))
# Esto es importante: no estamos reutilizando los mismos ejemplos con los que entrenamos el cross-encoder.


pairs = list(zip(silver['premise'], silver['hypothesis']))
# aunque MNLI realmente contiene las etiquetas de esos 40.000 ejemplos, en este experimento Alammar va a hacer 
# como si no las conociéramos. Es una simulación de datos unlabeled.
# zip() empareja cada premise con su hypothesis
# list lo convierte en uan lista
    # [
    #    (premise_1, hypothesis_1),
    #    (premise_2, hypothesis_2),
    #    (premise_3, hypothesis_3),
    #    ...
    #]

**Step 3:** Label new sentence pairs with the fine-tuned cross-encoder (silver dataset)

In [4]:
# el cross-encoder convierte los 40.000 pares sin etiquetar en el Silver dataset etiquetado por la máquina.

import numpy as np

# Label the sentence pairs using our fine-tuned cross-encoder
output = cross_encoder.predict(pairs, apply_softmax=True, show_progress_bar=True)
# El cross-encoder procesa cada par conjuntamente y, como lo configuramos con num_labels=2 -> produce 2 valores por cada par
    # Probabilidad de clase 0
    # Probabilidad de clase 1
# softmax -> convierte los 2 logits en valores que suman 1
silver = pd.DataFrame(
    {
        "sentence1": silver["premise"],
        "sentence2": silver["hypothesis"],
        "label": np.argmax(output, axis=1) # devuelve la posición de la probabilidad más alta -> si es primera dimensión:0 si no: 1
    }
)
# Creamos un DataFrame de pandas donde 
    # silver["premise"]        ← del dataset MNLI
    # silver["hypothesis"]    ← del dataset MNLI
    # np.argmax(output, axis=1)  ← predicciones del cross-encoder


Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

**Step 4:** Train a bi-encoder (SBERT) on the extended dataset (gold + silver dataset)

In [5]:
# Combine gold + silver
data = pd.concat([gold, silver], ignore_index=True, axis=0)
# concatenamos los dos DF de panadas gold y silver 
# [gold, silver] -> el orden es imortante para keep="first"
# axis=0 -> añadir filas, una debajo de otra.
# ignore_index=True -> hace que pandas cree un índice nuevo. en lugar de conservar los índices originales de Gold y Silver

data = data.drop_duplicates(subset=['sentence1', 'sentence2'], keep="first")
# Quitamos pares duplicados
# pandas mira exclusivamente la combinación: (sentence1, sentence2)
# keep="first" -> Si se diera el caso de que esté duplicado -> mantiene el primero -> gold

train_dataset = Dataset.from_pandas(data, preserve_index=False)
# Pasamos de pandas a Hugging Face Dataset (dataset.Dataset)
# preserve_index=False -> evita que el índice de pandas se convierta en una columna adicional del Dataset.

In [6]:
# Evaluamos como antes
# crea el mismo evaluador con STS-B para poder medir después si Augmented SBERT ha mejorado el embedding model.

from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [7]:
# entrenar el modelo de embeddings final con el dataset ampliado Gold + Silver.

from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define model
embedding_model = SentenceTransformer('bert-base-uncased') # bert base

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="augmented_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset, # datos concatenados de gold + silver
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.


Step,Training Loss
100,0.214200
200,0.156200
300,0.140000
400,0.142400
500,0.138800
600,0.135600
700,0.134500
800,0.131200
900,0.133000
1000,0.129600


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

TrainOutput(global_step=1563, training_loss=0.13938077382376313, metrics={'train_runtime': 297.0976, 'train_samples_per_second': 168.288, 'train_steps_per_second': 5.261, 'total_flos': 0.0, 'train_loss': 0.13938077382376313, 'epoch': 1.0})

In [8]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.6983330730059331,
 'spearman_cosine': 0.7094430459031037,
 'pearson_manhattan': 0.7111450197587281,
 'spearman_manhattan': 0.7094173394796922,
 'pearson_euclidean': 0.7107610645978151,
 'spearman_euclidean': 0.7089976915036289,
 'pearson_dot': 0.6665355451053444,
 'spearman_dot': 0.6681310547965276,
 'pearson_max': 0.7111450197587281,
 'spearman_max': 0.7094430459031037}

In [9]:
trainer.accelerator.clear()
# sirve para limpiar recursos que está manteniendo Hugging Face Accelerate después del entrenamiento, 
# especialmente referencias a objetos que pueden estar ocupando GPU/VRAM

[]

**Step 5**: Evaluate without silver dataset / Ahora Sólo con los 10000 de oro

In [10]:
# Combine gold + silver
data = pd.concat([gold], ignore_index=True, axis=0)
data = data.drop_duplicates(subset=['sentence1', 'sentence2'], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

# Define model
embedding_model = SentenceTransformer('bert-base-uncased')

# Loss function
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="gold_only_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.
C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Step,Training Loss
100,0.227400
200,0.172700
300,0.160900


TrainOutput(global_step=313, training_loss=0.18619733572767946, metrics={'train_runtime': 58.0197, 'train_samples_per_second': 172.355, 'train_steps_per_second': 5.395, 'total_flos': 0.0, 'train_loss': 0.18619733572767946, 'epoch': 1.0})

In [11]:
# Evaluate our trained model
evaluator(embedding_model)

{'pearson_cosine': 0.6422198071177381,
 'spearman_cosine': 0.6661691975724867,
 'pearson_manhattan': 0.6429133776077132,
 'spearman_manhattan': 0.6576604764055995,
 'pearson_euclidean': 0.6423739047291388,
 'spearman_euclidean': 0.65689668228487,
 'pearson_dot': 0.5883497714191555,
 'spearman_dot': 0.5930083873619857,
 'pearson_max': 0.6429133776077132,
 'spearman_max': 0.6661691975724867}

Compared to using both the silver and gold datasets, using only the gold dataset reduces the performance of the model!

⚠️ **VRAM Clean-up**
* `Restart` the notebook in order to clean-up memory if you move on to the next training example.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

## **Unsupervised Learning**

### Tranformer-based Denoising AutoEncoder (TSDAE)

In [2]:
# Esta línea prepara una dependencia que necesita el proceso de TSDAE para manipular el texto antes de "dañarlo":

# Download additional tokenizer

import nltk
# nltk -> Natural Language Toolkit, una librería clásica de NLP

nltk.download('punkt')
# punkt -> recurso de NLTK utilizado para tokenización, especialmente para detectar límites de frases y separar texto 
# de forma lingüísticamente razonable.
# Lo descarga porque DenoisingAutoEncoderDataset necesita herramientas de tokenización para el proceso de denoising que viene después.
# Importante: no es el tokenizer de BERT.

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\srmjf\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

```text
NLTK / punkt
    ↓
preparación / manipulación del texto
para generar la versión dañada

Tokenizer de BERT
    ↓
convierte posteriormente el texto
en tokens / IDs
    ↓
Transformer
```

In [3]:
# Este bloque construye exactamente el dataset que necesita TSDAE: para cada frase original crea una versión dañada y guarda ambas juntas.

from tqdm import tqdm
from datasets import Dataset, load_dataset
from sentence_transformers.datasets import DenoisingAutoEncoderDataset

# Create a flat list of sentences
mnli = load_dataset("glue", "mnli", split="train").select(range(25_000))
flat_sentences = mnli["premise"] + mnli["hypothesis"]
# Seleccionamos 25.000 filas, pero cada fila contiene premise y hypothesis, por lo que: 25000 + 25000 = 50000
# Simulamos un corpus de texto sin etiquetar


# Add noise to our input data
damaged_data = DenoisingAutoEncoderDataset(list(set(flat_sentences)))
# set(flat_sentences)-> elimina frases duplicadas. 
# list(...) vuelve a convertir el conjunto en una lista.
# DenoisingAutoEncoderDataset prepara cada frase para producir algo conceptualmente como:
    # ORIGINAL: "The capital of the Netherlands is Amsterdam"
    # DAMAGED:  capital Netherlands is Amsterdam
# damaged_data contiene ejemplos con las dos versiones:
    # damaged_data[0].texts[0] -> Frase dañada del ejemplo 0
    # damaged_data[0].texts[1] -> Frase original del ejemplo 1



# Create dataset / Construimos el dataset explícitamente
train_dataset = {"damaged_sentence": [], "original_sentence": []}
# Inicialmente tenemos dos listas vacías en un diccionario
for data in tqdm(damaged_data):
    train_dataset["damaged_sentence"].append(data.texts[0]) # guarda versión dañada
    train_dataset["original_sentence"].append(data.texts[1]) # guarda versión original
train_dataset = Dataset.from_dict(train_dataset)
# convierte nuestro diccionario Python en un datasets.Dataset de Hugging Face, como ya hemos visto varias veces.

100%|██████████| 48353/48353 [00:09<00:00, 5054.51it/s]


In [6]:
train_dataset[15]

{'damaged_sentence': 'taken who unfairly often.',
 'original_sentence': 'His place was taken by another person who thought people were charged unfairly often.'}

In [ ]:
# # Choose a different deletion ratio
# flat_sentences = list(set(flat_sentences))
# damaged_data = DenoisingAutoEncoderDataset(
#     flat_sentences,
#     noise_fn=lambda s: DenoisingAutoEncoderDataset.delete(s, del_ratio=0.6)
# )

In [7]:
# Creamos nuestro evaluator. como antes

from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# Create an embedding similarity evaluator for stsb
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [ ]:
from sentence_transformers import models, SentenceTransformer

# Create your embedding model
word_embedding_model = models.Transformer('bert-base-uncased')
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), 'cls')
embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

In [ ]:
from sentence_transformers import losses

# Use the denoising auto-encoder loss
train_loss = losses.DenoisingAutoEncoderLoss(
    embedding_model, tie_encoder_decoder=True
)
train_loss.decoder = train_loss.decoder.to("cuda")

In [ ]:
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# Define the training arguments
args = SentenceTransformerTrainingArguments(
    output_dir="tsdae_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100,
)

# Train model
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)
trainer.train()

In [ ]:
# Evaluate our trained model
evaluator(embedding_model)

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()